# Multitemporal Land Cover Mapping & HMM Consistency Analysis (Model Transfer + HLS Example)

Demonstrates a complete 2-period **Model Transfer** workflow using:
- **NASA HLS v2.0** (`HLS_L30` historical + `HLS` baseline) with CDF histogram matching
- Single training dataset on `baseline` ? single `FeatureExtraction.stratified_split()`
- Soft (`MULTIPROBABILITY`) Random Forest with scheme-named `prob_<Class_Name>` bands & optional `log_loss`
- **HMM / Viterbi Post-Classification Consistency Analysis** (`ConsistencyAnalyzer`) & pre/post accuracy comparison

In [ ]:
import os
import sys
import ee
import geemap
import pandas as pd

repo_root = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, os.path.join(repo_root, 'src'))

import luma_ge
from luma_ge import PeriodConfig, MultitemporalAnalysis, ConsistencyAnalyzer
from luma_ge.accuracy import thematic_accuracy

assert luma_ge.auto_initialize(), 'Earth Engine initialization failed!'

## 1. Define AOI, Scheme, Training Data, and Periods (2015 HLS vs 2025 HLS)

In [ ]:
# Replace with your project assets/shapefiles
aoi = ee.FeatureCollection('projects/ee-agilfahrezy60/assets/Sesi1_AOI_Kambera_NTT')
baseline_roi = ee.FeatureCollection('projects/ee-agilfahrezy60/assets/Sesi1_Data_Latih_Kambera')
baseline_val = ee.FeatureCollection('projects/ee-agilfahrezy60/assets/Sesi2_Data_Penguji_Akurasi')
scheme_df = pd.read_csv(os.path.join(repo_root, 'data', 'MODUL2_Skema_Klasifikasi_Kambera.csv'))

mt = MultitemporalAnalysis(
    aoi=aoi,
    periods=[
        PeriodConfig(label='historical', start_date=2015, end_date=2015, sensor='HLS_L30', cloud_cover=40),
        PeriodConfig(label='baseline', start_date=2025, end_date=2025, sensor='HLS', cloud_cover=30),
    ],
    reference_period='baseline',
    histogram_match=True,
    scale=30,
)

## 2. Run End-to-End Multitemporal Classification (`MULTIPROBABILITY` + Optional Log Loss)

In [ ]:
result = mt.run_pipeline(
    training_sources={'historical': None, 'baseline': baseline_roi},
    landcover_df=scheme_df,
    predictor_config={
        'individual_predictors': {
            'elevation': True,
            'slope': True,
            'aspect': False,
            'spectral_indices': ['NDVI', 'MNDWI', 'NDBI'],
        }
    },
    validation_by_period={'baseline': baseline_val, 'historical': None},
    class_field='ID_Kelas',
    class_name_property='KelasTupla',
    use_class_ids=True,
    min_samples=5,
    classification_type='soft',
    soft_mode='multiprobability',
    ntrees=100,
    include_log_loss=True,
    verbose=True,
)

eval_base = result['classification']['model_evaluation']['baseline']
print('Mode used:', result['mode'])
print('Extraction type:', result['feature_extraction']['extraction_type'])
print('Baseline Model OA:', eval_base['overall_accuracy'], '| Log Loss:', eval_base.get('log_loss'))
print('Probability bands:', result['classification']['prob_bands']['baseline'])

## 3. Post-Classification Consistency Analysis (`ConsistencyAnalyzer`)

In [ ]:
ca = ConsistencyAnalyzer()
class_ids = sorted(int(c) for c in scheme_df['ID'].tolist())

# Forbidden irreversible reversions (e.g., Built-up 7 -> Forest 1, 2 or Cropland 5, 6)
forbidden = {7: {1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 5: 0.0, 6: 0.0}, 8: {1: 0.0}}
matrix = {
    fid: {tid: (1.0 if fid == tid else forbidden.get(fid, {}).get(tid, 0.8)) for tid in class_ids}
    for fid in class_ids
}
val_mat = ca.validate_transition_matrix(matrix, class_ids=class_ids)

geom = aoi.geometry()
past_raw = result['classification']['maps']['historical']
future_ref = result['classification']['maps']['baseline']

ct_raw = ca.cross_tabulate(past_raw, future_ref, val_mat, geom, scale=30, class_names=scheme_df, class_ids=class_ids)
print('Raw total px:', ct_raw['total_pixels'], '| Inconsistent px:', ct_raw['inconsistent_pixels'])
display(ct_raw['inconsistent_transitions'])

# Primary HMM correction anchored to baseline with 3x3 spatial smoothing
hmm_out = ca.correct_inconsistencies(
    past_map=past_raw,
    future_map=future_ref,
    consistency_matrix=val_mat,
    aoi=geom,
    method='hmm',
    anchor_reference=True,
    spatial_smoothing=True,
    spatial_weight=0.25,
    scale=30,
    class_ids=class_ids,
    class_names=scheme_df,
    prob_band_map=result['classification']['prob_bands']['historical'],
)
print('Corrected px:', hmm_out['corrected_pixel_count'], '| Remaining inconsistent:', hmm_out['remaining_inconsistent_count'])

## 4. Map Comparison & Pre/Post Accuracy Assessment

In [ ]:
# If you have historical ground truth (hist_val), compare accuracy before vs after correction:
# taa = thematic_accuracy()
# _, raw_acc = taa.run_accuracy_assessment(past_raw, hist_val, class_property='ID_Kelas', scale=30)
# _, corr_acc = taa.run_accuracy_assessment(hmm_out['corrected_map'], hist_val, class_property='ID_Kelas', scale=30)
# print('Accuracy Delta:', taa.compare_accuracy(raw_acc, corr_acc))

Map = geemap.Map()
vis = {
    'bands': ['classification'],
    'min': int(scheme_df['ID'].min()),
    'max': int(scheme_df['ID'].max()),
    'palette': scheme_df.sort_values('ID')['Color Palette'].tolist(),
}
Map.addLayer(past_raw.select('classification').clip(geom), vis, '2015 Historical (Raw)')
Map.addLayer(hmm_out['corrected_map'].clip(geom), vis, '2015 Historical (HMM Consistent)')
Map.addLayer(future_ref.select('classification').clip(geom), vis, '2025 Baseline')
Map.centerObject(aoi, 12)
Map